# RSNA Knee — submission (inference only, Internet OFF)
学習は `02_cnn_embed` で行い、保存した重みとモデルをKaggle Datasetから読む。`src` も Dataset から。

In [ ]:
import os, sys, warnings
os.environ['PYTHONWARNINGS'] = 'ignore'; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch, joblib
from pathlib import Path

def find(name):
    # Kaggleのマウント位置(/kaggle/input/<slug> または .../datasets/<user>/<slug>)の違いを吸収
    for base in [Path('/kaggle/input'), Path('/kaggle/input/datasets')]:
        if not base.exists(): continue
        for d in base.iterdir():
            if d.name == name: return d
            if d.is_dir() and d.name not in ('competitions', 'datasets'):
                for s in d.iterdir():
                    if s.name == name: return s
    raise FileNotFoundError(f'{name} not found. /kaggle/input: {[p.name for p in Path("/kaggle/input").iterdir()]}')

SRC, MODEL = find('rsna-knee-src'), find('rsna-knee-model')
sys.path.insert(0, str(SRC))
import cnn_features as cf
from pseudo_labels import LABELS
_name = 'rsna-knee-abnormality-detection'
COMP = next((d for d in [Path('/kaggle/input/competitions')/_name, Path('/kaggle/input')/_name] if (d/'test.csv').exists()), None)
assert COMP, 'test.csv not found'
print('COMP', COMP, '| MODEL', MODEL)

In [ ]:
test = pd.read_csv(COMP/'test.csv'); teseries = pd.read_csv(COMP/'test_series.csv')
print(test.shape, teseries.shape)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = cf.load_model(dev, weights_path=MODEL/'resnet18_imagenet.pth')
models = joblib.load(MODEL/'models.joblib')
Xte = cf.extract(test, teseries, COMP/'test_series', model, dev)

In [ ]:
import cv_eval
sub = cv_eval.predict_models(models, Xte, LABELS, test['StudyInstanceUID'])
assert sub.shape[0] == len(test) and sub[LABELS].notna().all().all()
sub.to_csv('submission.csv', index=False); print(sub.shape); sub.head()